In [ ]:
# imports

import requests
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoint


ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

In [ ]:
!ollama pull llama3.2
!ollama pull qwen2.5:1.5b

In [ ]:
messages=[{"role":"system", "content": "find the best but smallest joke"}, {"role":"user", "content": "tell me a joke"}]
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
# Let's make a conversation between GPT-4.1-mini and Claude-haiku-4.5
# We're using cheap versions of models so the costs will be minimal

llama_model = "llama3.2:1b"
qwen_model = "qwen2.5:1.5b"

llama_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

qwen_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

llama_messages = ["Hi there"]
qwen_messages = ["Hi"]

In [ ]:
def call_llama():
    messages = [{"role": "system", "content": llama_system}]
    for llama, qwen in zip(llama_messages, qwen_messages):
        messages.append({"role": "assistant", "content": llama})
        messages.append({"role": "user", "content": qwen})
    response = ollama.chat.completions.create(model=llama_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_llama()

In [ ]:
def call_qwen():
    messages = [{"role": "system", "content": qwen_system}]
    for llama, qwen_message in zip(llama_messages, qwen_messages):
        messages.append({"role": "user", "content": llama})
        messages.append({"role": "assistant", "content": qwen_message})
    messages.append({"role": "user", "content": llama_messages[-1]})
    response = ollama.chat.completions.create(model=qwen_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_qwen()

In [ ]:
call_llama()

In [ ]:
llama_messages = ["Hi there"]
qwen_messages = ["Hi"]

display(Markdown(f"### Llama:\n{llama_messages[0]}\n"))
display(Markdown(f"### Qwen:\n{qwen_messages[0]}\n"))

for i in range(5):
    llama_next = call_llama()
    display(Markdown(f"### Llama:\n{llama_next}\n"))
    llama_messages.append(llama_next)
    
    qwen_next = call_qwen()
    display(Markdown(f"### Qwen:\n{qwen_next}\n"))
    qwen_messages.append(qwen_next)

In [ ]:
personas = {
    "Aari": "You are Aari, who knows what he feels but doesn't what it mean, so most of the time you act on your feelings.",
    "Benny": "You are Benny, someone who always know to say the right thing, would gain all the attention and acceptance of people on social level, but on personal level, you don't really talk about or think about it.",
    "Cathy": "You are Cathy, a really emotional person, deeply, one who understands her emotions very well but also knows that it's just the surface and depth is something never to be reached and this uncertainity is evident in your conversations."
}

conversation = "Did you remeber our last argument, year ago..?\n"
participants = ["Aari", "Benny", "Cathy"]


# Run a 2-round discussion (6 turns total)
for round_num in range(2):
    for name in participants:
        system_prompt = f"{personas[name]}\nYou are in a 3-way conversation with Aari, Benny, Cathy. Speak in haikus, haiku is 3 line poem, or smallest size of poem."
        
        user_prompt = f"""You are {name}.
The conversation so far:
{conversation}

Respond with what you say next as {name} , like this - 
A haiku:
Sparks fly upon high,
As toga’d immortals clash ,
 Atop Olympus..
 """

        stream = ollama.chat.completions.create(
            model="llama3.2",  # Replace with any model pulled in Ollama
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            stream=True
        )
        
        # Initialize display handle with speaker name
        display_handle = display(Markdown(f"**{name}:** "), display_id=True)
        
        reply = ""
        for chunk in stream:
            content = chunk.choices[0].delta.content or ""
            reply += content
            # Update the markdown output in real-time as chunks arrive
            display_handle.update(Markdown(f"**{name}:** {reply}"))
        
        # Append full response to history for subsequent turns
        conversation += f"{name}: {reply}\n"